In [1]:
import tensorflow as tf
import numpy as np
import json

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [5]:
import os

print(os.getcwd())

C:\Disk D\Year 2 Semester 1\Year2Semester1\Year2Semester1\IT2011 AI & ML\AI ML Project\Waste-Classification-AI\notebook


In [6]:
import os

print(os.listdir("../models"))

['final_main_category_model.keras', 'final_organic_model.keras', 'final_recyclable_model.keras', 'hazardous_model.keras', 'main_category_model.keras', 'non_recyclable_model.keras']


In [8]:
import tensorflow as tf


main_model = tf.keras.models.load_model(
    "../models/final_main_category_model.keras",
    compile=False
)


hazardous_model = tf.keras.models.load_model(
    "../models/hazardous_model.keras",
    compile=False
)


organic_model = tf.keras.models.load_model(
    "../models/final_organic_model.keras",
    compile=False
)


recyclable_model = tf.keras.models.load_model(
    "../models/final_recyclable_model.keras",
    compile=False
)


non_recyclable_model = tf.keras.models.load_model(
    "../models/non_recyclable_model.keras",
    compile=False
)


print("All models loaded successfully")

All models loaded successfully


In [10]:
import json


with open("../models/class_labels.json", "r") as f:
    labels = json.load(f)


print("Labels loaded")

Labels loaded


In [11]:
def prepare_image(img_path):

    img = image.load_img(
        img_path,
        target_size=(224,224)
    )


    img_array = image.img_to_array(img)


    img_array = np.expand_dims(
        img_array,
        axis=0
    )


    img_array = preprocess_input(
        img_array
    )


    return img_array

In [24]:
def predict_waste(img_path):

    img = prepare_image(img_path)


    # -------------------------
    # LEVEL 1
    # -------------------------

    main_pred = main_model.predict(img)


    main_index = np.argmax(main_pred)

    main_conf = np.max(main_pred)


    category = labels["main"][str(main_index)]


    print("\n========== RESULT ==========")

    print(
        "Main Category:",
        category
    )

    print(
        "Category Confidence:",
        round(main_conf*100,2),
        "%"
    )


    # -------------------------
    # SELECT LEVEL 2 MODEL
    # -------------------------


    if category == "Hazardous":

        model = hazardous_model
        label_name = "hazardous"


    elif category == "Organic":

        model = organic_model
        label_name = "organic"


    elif category == "Recyclable":

        model = recyclable_model
        label_name = "recyclable"


    else:

        model = non_recyclable_model
        label_name = "non_recyclable"



    # -------------------------
    # LEVEL 2
    # -------------------------

    detail_pred = model.predict(img)


    detail_index = np.argmax(detail_pred)


    detail_conf = np.max(detail_pred)


    waste_type = labels[label_name][str(detail_index)]


    print(
        "Specific Waste:",
        waste_type
    )


    print(
        "Specific Confidence:",
        round(detail_conf*100,2),
        "%"
    )


    print("============================")


    return {
    "category": category,
    "category_confidence": float(round(main_conf * 100, 2)),
    "waste_type": waste_type,
    "waste_confidence": float(round(detail_conf * 100, 2))
}

In [25]:
import os
import random


dataset_paths = [

    "../dataset/hazardous_dataset",
    "../dataset/organic_dataset",
    "../dataset/recyclable_dataset",
    "../dataset/non_recyclable_dataset"

]


all_images = []


for dataset in dataset_paths:

    for root, dirs, files in os.walk(dataset):

        for file in files:

            if file.lower().endswith(
                (".jpg",".jpeg",".png",".webp")
            ):

                all_images.append(
                    os.path.join(root,file)
                )


print("Total test images found:", len(all_images))


test_image = random.choice(all_images)


print("\nTesting:")
print(test_image)


result = predict_waste(test_image)


print("\nFinal Output:")
print(result)

Total test images found: 2917

Testing:
../dataset/hazardous_dataset\e-waste\an-electronics-e-waste-recycling-collection-area-the-collection-is-APK6DT - Copy.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step

========== RESULT ==========
Main Category: Hazardous
Category Confidence: 97.92 %
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Specific Waste: e-waste
Specific Confidence: 97.91 %

Final Output:
{'category': 'Hazardous', 'category_confidence': 97.91999816894531, 'waste_type': 'e-waste', 'waste_confidence': 97.91000366210938}
